### Model Subclassing & Custom Training Loop Using Reuters Dataset

In [2]:
#importing libraries

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import time

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Softmax

In [9]:
#defining the custom layers and model

class MyLayer(Layer):

    def __init__(self, units, input_dims):
        super(MyLayer, self).__init__()
        self.w = self.add_weight(shape=(input_dims, units),
                    initializer="random_normal")
        self.b = self.add_weight(shape=(units, ),
                    initializer="zeros")

    def call(self, inputs):
        return tf.matmul(inputs, self.w) + self.b


class MyDropout(Layer):

    def __init__(self, rate):
        super(MyDropout, self).__init__()
        self.rate = rate

    def call(self, inputs):
        return tf.nn.dropout(inputs, rate=self.rate)  


class MyModel(Model):

    def __init__(self, units_1, input_dim_1, units_2, units_3):
        super(MyModel, self).__init__()
        self.layer_1 = MyLayer(units_1, input_dim_1)  
        self.layer_2 = MyLayer(units_2, units_1)
        self.layer_3 = MyLayer(units_3, units_2)
        self.dropout_1 = MyDropout(0.5)
        self.dropout_2 = MyDropout(0.5)
        self.softmax = Softmax()

    def call(self, inputs):
        #define forward pass
        x = self.layer_1(inputs)
        x = tf.nn.relu(x)
        x = self.dropout_1(x)
        x = self.layer_2(x)
        x = tf.nn.relu(x)
        x = self.dropout_2(x)
        x = self.layer_3(x)
        x = tf.nn.relu(x)
        return self.softmax(x)


In [10]:
#instantiating the model object

model = MyModel(64, 10000, 64, 46)
print(model(tf.ones((1, 10000))))
model.summary()

tf.Tensor(
[[0.01657389 0.01657389 0.02819896 0.01657389 0.01657389 0.01657389
  0.01719997 0.01657389 0.01657389 0.01657389 0.01657389 0.01657389
  0.01657389 0.01657389 0.03588191 0.02143023 0.01657389 0.02011918
  0.02162978 0.02734404 0.01657389 0.03370906 0.01657389 0.01657389
  0.02665621 0.01657389 0.01657389 0.01657389 0.05641206 0.02148852
  0.01657389 0.04779866 0.01657389 0.01657389 0.02157116 0.02212267
  0.0369702  0.01657389 0.03313976 0.01657389 0.03062619 0.01657389
  0.01657389 0.01657389 0.02886027 0.021346  ]], shape=(1, 46), dtype=float32)


Model: "my_model_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_layer_9 (MyLayer)            │ ?                      │       640,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_10 (MyLayer)           │ ?                      │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_11 (MyLayer)           │ ?                      │         2,990 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dropout_6 (MyDropout)        │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dropout_7 (MyDropout)        │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_3 (Softmax)             │ ?                      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 647,214 (2.47 MB)

 Trainable params: 647,214 (2.47 MB)

 Non-trainable params: 0 (0.00 B)